# [Introduction to Data Science](http://datascience-intro.github.io/1MS041-2026/)    
## 1MS041, 2026 
&copy;2026 Raazesh Sainudiin, Benny Avelin. [Attribution 4.0 International     (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

# ProbSS 8 — From PCA to $K$-means

## What you will do

You will scale data without leakage, use PCA for reconstruction, and use
$K$-means to find groups. You will compare initialisations and choices of $K$,
measure time and storage, and finish with a simple recommendation baseline.

$K$-means minimises squared distance to cluster centres. It does not promise
that the clusters are scientifically meaningful, so interpretation remains
part of the task.


In [ ]:
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from pathlib import Path

def course_data(filename):
    candidates = (
        Path("data") / filename,
        Path("master/jp/data") / filename,
    )
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find {filename}. Run this notebook from master/jp "
        "or from the repository root."
    )

RANDOM_STATE = 2026


## 1. Keep development, validation, and test data separate

The digit images are bundled with scikit-learn. Labels are used only to balance the splits; they are not supplied to PCA or K-means. Scaling and PCA are fitted on training data, K-means diagnostics use validation data, and the final test set is used once after the procedure is fixed.


In [ ]:
digits = load_digits()
X = digits.data.astype(float)
y = digits.target.astype(int)

X_development, X_test, y_development, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)
X_train, X_validation, y_train, y_validation = train_test_split(
    X_development,
    y_development,
    test_size=0.25,
    stratify=y_development,
    random_state=RANDOM_STATE,
)

scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_validation_scaled = scaler.transform(X_validation)

pca = PCA(n_components=20, random_state=RANDOM_STATE).fit(X_train_scaled)
Z_train = pca.transform(X_train_scaled)
Z_validation = pca.transform(X_validation_scaled)

print("train, validation, test:", X_train.shape, X_validation.shape, X_test.shape)
print("20-component explained variance:", pca.explained_variance_ratio_.sum())


PCA finds directions that preserve variation and supports reconstruction. K-means partitions points to reduce within-cluster squared distances. Either can be useful without serving the same purpose.


## 2. Why $K$-means initialisation matters


In [ ]:
initialisation_rows = []
for seed in range(6):
    model = KMeans(
        n_clusters=10,
        init="k-means++",
        n_init=1,
        random_state=seed,
    ).fit(Z_train)
    validation_distortion = np.mean(
        np.min(model.transform(Z_validation) ** 2, axis=1)
    )
    initialisation_rows.append(
        {
            "seed": seed,
            "training objective per row": model.inertia_ / len(Z_train),
            "validation distortion": validation_distortion,
        }
    )

initialisation_results = pd.DataFrame(initialisation_rows)
initialisation_results


Different initial centres can lead to different local optima. A fixed random state makes a run reproducible; multiple starts make the procedure more robust.


## 3. Choose $K$ for a reason


In [ ]:
k_values = [5, 8, 10, 12, 15]
k_rows = []
for k in k_values:
    model = KMeans(
        n_clusters=k,
        n_init=10,
        random_state=RANDOM_STATE,
    ).fit(Z_train)
    k_rows.append(
        {
            "K": k,
            "training objective per row": model.inertia_ / len(Z_train),
            "validation distortion": np.mean(
                np.min(model.transform(Z_validation) ** 2, axis=1)
            ),
        }
    )
k_results = pd.DataFrame(k_rows)
k_results


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(k_results["K"], k_results["validation distortion"], marker="o")
ax.set(
    xlabel="number of clusters K",
    ylabel="validation squared-distance distortion",
    title="Held-out distortion decreases as the codebook grows",
)
ax.grid(alpha=0.2)
plt.show()


Distortion alone generally falls as $K$ increases, so its minimum is not a neutral discovery of a true number of groups. Here we fix $K=10$ because the stated engineering purpose is a ten-prototype codebook for the ten digit symbols. In another application, domain meaning, stability, resource limits, and downstream usefulness must enter the decision.


## 4. Refit the chosen method and test it once

The resource table below separates three quantities that must not be conflated:

- measured operation time (scaling and PCA each include fitting plus development/test transformation; K-means includes fitting);
- the stored development output of that stage; and
- the main fitted numerical arrays needed by the stage.

These counts exclude Python-object overhead and temporary peak working memory, so they are reproducible lower-level comparisons rather than complete process-memory measurements.


In [ ]:
start = perf_counter()
final_scaler = StandardScaler().fit(X_development)
X_development_scaled = final_scaler.transform(X_development)
X_test_scaled = final_scaler.transform(X_test)
scale_seconds = perf_counter() - start

start = perf_counter()
final_pca = PCA(n_components=20, random_state=RANDOM_STATE).fit(
    X_development_scaled
)
Z_development = final_pca.transform(X_development_scaled)
Z_test = final_pca.transform(X_test_scaled)
pca_seconds = perf_counter() - start

start = perf_counter()
final_kmeans = KMeans(
    n_clusters=10,
    n_init=20,
    random_state=RANDOM_STATE,
).fit(Z_development)
kmeans_seconds = perf_counter() - start

test_distortion = np.mean(
    np.min(final_kmeans.transform(Z_test) ** 2, axis=1)
)
reconstructed_test = final_pca.inverse_transform(Z_test)
reconstruction_mse = np.mean((X_test_scaled - reconstructed_test) ** 2)

print(f"Final held-out K-means distortion: {test_distortion:.3f}")
print(f"Final held-out PCA reconstruction MSE: {reconstruction_mse:.3f}")


In [ ]:
development_cluster_labels = final_kmeans.labels_

scaler_parameter_bytes = final_scaler.mean_.nbytes + final_scaler.scale_.nbytes
pca_parameter_bytes = final_pca.mean_.nbytes + final_pca.components_.nbytes
kmeans_parameter_bytes = final_kmeans.cluster_centers_.nbytes

resource_table = pd.DataFrame(
    {
        "stage": ["scaling", "PCA", "K-means"],
        "measured operation seconds": [scale_seconds, pca_seconds, kmeans_seconds],
        "development output MB": [
            X_development_scaled.nbytes / 1e6,
            Z_development.nbytes / 1e6,
            development_cluster_labels.nbytes / 1e6,
        ],
        "main fitted arrays MB": [
            scaler_parameter_bytes / 1e6,
            pca_parameter_bytes / 1e6,
            kmeans_parameter_bytes / 1e6,
        ],
    }
)
resource_table


In [ ]:
centres_scaled = final_pca.inverse_transform(final_kmeans.cluster_centers_)
centres = final_scaler.inverse_transform(centres_scaled).reshape(-1, 8, 8)

fig, axes = plt.subplots(2, 5, figsize=(9, 4))
for index, ax in enumerate(axes.ravel()):
    ax.imshow(centres[index], cmap="gray_r")
    ax.set_title(f"cluster {index}")
    ax.axis("off")
fig.suptitle("K-means prototypes in the original pixel coordinates")
plt.tight_layout()
plt.show()


The prototypes can be inspected for meaning, but cluster indices are arbitrary and need not correspond one-to-one with digit labels. A mixture-model interpretation requires additional distributional assumptions and does not guarantee recovery of latent labels.


## 5. A simple recommender with a time split

To make a genuine future-period comparison, use one cutoff for the entire ratings table. Every rating strictly before the cutoff is training data; every rating at or after the cutoff is test data. Keeping all rows tied at the boundary in the test period prevents an arbitrary `idxmax` tie-break from placing simultaneous records on both sides.

This target differs from leave-one-rating-out evaluation. It asks how baselines fitted to the earlier global period perform in a later global period, including later users and items that may not appear in training.


In [ ]:
ratings = pd.read_csv(course_data("ratings.csv"))
ratings = ratings.sort_values(["timestamp", "userId", "movieId"]).reset_index(drop=True)

# Select a global boundary from timestamps only, without inspecting ratings.
# All observations tied at the boundary go to the test period.
cutoff_timestamp = int(
    ratings["timestamp"].quantile(0.80, interpolation="lower")
)
ratings_train = ratings.loc[ratings["timestamp"] < cutoff_timestamp].copy()
ratings_test = ratings.loc[ratings["timestamp"] >= cutoff_timestamp].copy()

assert len(ratings_train) > 0 and len(ratings_test) > 0
assert ratings_train["timestamp"].max() < ratings_test["timestamp"].min()
assert not ratings_train["timestamp"].eq(cutoff_timestamp).any()

global_mean = ratings_train["rating"].mean()
item_stats = ratings_train.groupby("movieId")["rating"].agg(["mean", "count"])
shrinkage = 20.0
item_stats["score"] = (
    item_stats["count"] * item_stats["mean"] + shrinkage * global_mean
) / (item_stats["count"] + shrinkage)

global_prediction = np.full(len(ratings_test), global_mean)
item_prediction = (
    ratings_test["movieId"].map(item_stats["score"]).fillna(global_mean).to_numpy()
)
global_mae = np.mean(np.abs(ratings_test["rating"] - global_prediction))
item_mae = np.mean(np.abs(ratings_test["rating"] - item_prediction))

example_user = 1
seen = set(ratings_train.loc[ratings_train["userId"] == example_user, "movieId"])
top_unseen = item_stats.loc[~item_stats.index.isin(seen)].nlargest(5, "score")

cutoff_utc = pd.to_datetime(cutoff_timestamp, unit="s", utc=True)
cold_start_rows = int((~ratings_test["movieId"].isin(item_stats.index)).sum())
print("Global cutoff (assuming Unix seconds):", cutoff_utc)
print("Training and test rows:", len(ratings_train), len(ratings_test))
print("Later-period rows for unseen items:", cold_start_rows)
print(f"Global-mean global-time-holdout MAE: {global_mae:.3f}")
print(f"Shrunken item-mean holdout MAE:      {item_mae:.3f}")
print(f"Top unseen item IDs for user {example_user}:")
print(top_unseen[["score", "count"]])


This recommender is a baseline, not a personalised causal model. It assumes that ratings before the global cutoff are useful for ratings in the later period and that item averages transfer across users. It is affected by popularity and selection bias, gives little help for new items, and uses identifiers that still require careful handling. The later-period test rows are not IID merely because they occur after one cutoff.

Data-responsibility checklist:

- source/licence: the local ratings copy needs a verified MovieLens citation and reuse record before publication;
- privacy: user identifiers are pseudonymous, not automatically anonymous;
- integrity: use one global time cutoff, put every boundary-tied record in the later period, and assert strict temporal separation;
- resources: record fit time, feature dimension, score-table size, and candidate-set size;
- fairness and usefulness: inspect which users/items receive poor coverage and compare with the global baseline;
- communication: state that item IDs and cluster numbers are not semantic explanations.


In [ ]:
conclusion = (
    f"The 20-component digit representation used "
    f"{100 * Z_development.nbytes / X_development_scaled.nbytes:.1f}% of the "
    "dense development-array storage and supported a ten-prototype K-means "
    f"codebook with held-out distortion {test_distortion:.2f}. The clusters "
    "are optimisation groups, not verified natural classes. On the ratings "
    f"data, the shrunken item baseline changed global-time-holdout MAE from "
    f"{global_mae:.3f} to {item_mae:.3f}; this comparison does not resolve "
    "selection bias, cold start, temporal drift, or privacy."
)
print(conclusion)


## Recap

Before you finish, make sure you can:

1. Explain the effects of scaling, PCA dimension, initialisation, and $K$ on the stated task.
2. Report held-out distortion, reconstruction error, runtime, and array storage without treating any one number as sufficient.
3. Compare PCA and K-means by purpose.
4. Compare the recommender with the global baseline and name one user or item group for which it may fail.
5. Write a nontechnical conclusion and complete the data-responsibility checklist.
